# Functions utilized to clean data Tables
### This notebook is comprised of various functions used to clean and format the EMR data tables

Februrary 17th, 2025

Maxime Bouthillier

In [2]:
import pandas as pd
import os 
import glob 
from datetime import datetime

In [3]:
def readmission(dataframe, read_time): 

    '''
    This function will identify all instances of readmission within 
    the user defined timeframe (read_time). 

    This is accomplished in the following and somewhat convoluted manner:

        The function takes the given dataframe and iterates over all 
        unique subject_ids. If there are multiple entries, a subset dataframe 
        named "mult_entries" is then created and the new 'read_flag' 
        variable = 2 on all entries. If there is a singular entry, the new 
        readmission variable, 'read_flag', = 0. 

        A nested forloop is then utilized to iterate over the "mult_entries" 
        dataframe. An if statement is then imployed to check the time between the 
        previous discharge time to the current admission time. If this time delta is
        less than the user defined read_time, a readmission flag (=1) is appended
        to the previous entry, and a readmission flag of 2 is appeneded to the 
        current entry.

        All readmission flag entries of =2 is then removed from the dataset, as any 
        subject_id entries after the readmission priod is to be removed. Moreover, 
        Any additional subject_id entries with a time delta greater than the time
        delta is also removed (flaged with 2) as to not confuse the model structure
        given the high-dimensionality of the dataset.


    Note: the dataframe is filtered by earliest to the latest date. This ensures the
    funcationality of the nested forloop.
    '''

    # Unsure why but re-importing datetime allows for the use of the time.delta function
    import datetime 

    dataframe = dataframe.sort_values('admittime')    
    subjects = dataframe['subject_id'].unique()
    read_30d = []

    for j in subjects:                                                                                                                         

        if len(dataframe.loc[(dataframe['subject_id'] == j)]) >= 2 :                                                                              

            mult_entries = dataframe.loc[(dataframe['subject_id'] == j)].reset_index()                                                            
            dataframe.loc[(dataframe['subject_id'] == j), 'read_flag'] = 2                                                                            
            dataframe.loc[(dataframe['subject_id'] == j) & (dataframe['admittime'] == mult_entries['admittime'][0]), 'read_flag'] = 0               
        
            for k in range(len(mult_entries['admittime'])-1):
                if abs(mult_entries['dischtime'][k] - mult_entries['admittime'][k+1]) <= datetime.timedelta(days=read_time):                        
                    dataframe.loc[(dataframe['subject_id'] == j) & (dataframe['admittime'] == mult_entries['admittime'][k]) , 'read_flag'] = 1      
                    dataframe.loc[(dataframe['subject_id'] == j) & (dataframe['admittime'] != mult_entries['admittime'][k]) , 'read_flag'] = 2

        else:
            dataframe.loc[(dataframe['subject_id'] == j), 'read_flag'] = 0


    dataframe = dataframe.drop(dataframe[dataframe['read_flag'] == 2].index)  
     
    return(dataframe)


In [4]:
def as_datetime(dataframe, column):

    '''
    Converts the user specified columns' entries into datatime object. 
    This allows for future use of arithmetic operations. This returns the entire dataframe,
    not only the specified column.
    '''

    dataframe[column] = dataframe[column].apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S'))
    return dataframe

In [5]:
def subject_subset(dataframe, column):

    '''
    Subsets the user defined dataframe based on the provided column
    by the refined list of subjects available in the admission dataframe
    (admdf). Returns the cleaned subsetted dataframe. 

    IMPORTANT: This requires that the Admission.ipynb file be executed to produce
    the cleaned ADMISSIONS.csv.
    '''
    
    admdf = pd.read_csv('/Users/maxb/Library/CloudStorage/OneDrive-UniversityofWaterloo/Hospital Research/Datasets/Cleaned MIMIC-III Dataset/ADMISSIONS.csv')

    dataframe_new = pd.DataFrame()
    subjects = list(admdf['subject_id'])

    for i in subjects:
        if i in list(dataframe['subject_id']):

            admittime = admdf.loc[admdf['subject_id'] ==  i, 'admittime'].iloc[0]
            dischtime = admdf.loc[admdf['subject_id'] ==  i, 'dischtime'].iloc[0]

            dataframe_new = pd.concat([dataframe.loc[(dataframe['subject_id'] == i) & (dataframe[column] >= admittime) & 
                (dataframe[column] <= dischtime)], dataframe_new])

    dataframe = dataframe_new.fillna(0)

    return(dataframe)

In [ ]:
def subjectcheck(dataframe_1, dataframe_2):

    ''''''


    subjects = list(dataframe_1['subject_id'].unique())
    crosscheck = []

    for i in subjects:
        if i in list(dataframe_2['subject_id']):
            crosscheck.append(i)
    
    return(crosscheck)

In [ ]:
def check_nan(dataframe):

    '''Simple functino checking whether a NaN value is present within the given dataset'''

    for i in list(dataframe.columns):
        check_nan = dataframe[i].isnull().values.any()
        if check_nan == True:
            print(i, "NaN value found")
            raise SystemExit("Stopping execution")

In [8]:
# def caregiver(dataframe):

#     '''
#     Pairing the Caregiver ID (cgid) with the appropriate 
#     type of personal (Phycisian, Nurse, PSW, etc.,). This is
#     done pairing the cgid entry witht he corresponding entry 
#     from the cgid dataframe. 
#     '''

#     cgid_df = pd.read_csv("/Users/maxb/Library/CloudStorage/OneDrive-UniversityofWaterloo/Hospital Research/Datasets/MIMIC-III_demo/CAREGIVERS.csv")
#     dataframe_new = pd.DataFrame()

#     for i in dataframe['cgid']:
#         dataframe_new = pd.concat([dataframe.loc[(dataframe['cgid'] == i) & (dataframe[column] >= pd.Series.to_string(admdf.loc[admdf['subject_id'] ==  i, 'admittime'], index=False)) & 
#                 (dataframe[column] <= pd.Series.to_string(admdf.loc[admdf['subject_id'] ==  i, 'dischtime'], index=False))], dataframe_new])